# Xiangqi — resumable CPU batch collector

Chạy trên nhánh `perf/alphabeta-batch-resume`. Dataset **JSONL = 1 ván / 1 dòng**. Chỉ main process ghi lên Google Drive, checkpoint được cập nhật sau mỗi ván. Nếu Colab ngắt, chạy lại các cell và đặt cùng `RUN_NAME` để **tiếp tục các game_index còn thiếu**.

**Lưu ý:** Chỉ dùng Colab theo chính sách và giới hạn tài nguyên hợp lệ; Alpha-Beta thuần Python chạy trên CPU, không cần GPU.

## 1. Kiểm tra tài nguyên và tải source

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version.split()[0])
print('CPU vCPUs:', os.cpu_count())
print('Architecture:', platform.platform())

REPO_DIR = Path('/content/ML-Advance-Group')
BRANCH = 'perf/alphabeta-batch-resume'
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH,
        'https://github.com/Ko4rut/ML-Advance-Group.git', str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

BACKEND_DIR = REPO_DIR / 'project' / 'backend'
assert (BACKEND_DIR / 'src' / 'training' / 'batch_resume.py').exists()
print('Backend ready:', BACKEND_DIR)
print('Revision:', subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True,
).strip())

## 2. Unit tests + benchmark (chạy trước batch lớn)

Test luật đi, Alpha-Beta và checkpoint. Benchmark so sánh engine gốc với bản tối ưu trên các trạng thái giống nhau.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'unittest', 'discover',
    '-s', 'tests', '-p', 'test_*.py', '-v',
], cwd=str(BACKEND_DIR), check=True)

subprocess.run([
    sys.executable, '-u', '-m', 'src.evaluation.compare_alphabeta',
    '--depth', '2', '--positions', '3',
], cwd=str(BACKEND_DIR), check=True)

Dataset ghi **sau mỗi ván**, checkpoint ghi thêm `completed_indices`, `target_games` và cấu hình. Khi khởi động lại, chương trình đọc **JSONL thực tế** trước để xác định game_index đã xong; checkpoint có thể được phục hồi nếu lần trước bị ngắt giữa lúc cập nhật. Không ghi trùng seed/index.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = Path('/content/drive/MyDrive/XiangqiAI/datasets')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('Output folder:', SAVE_DIR)

## 4. Cấu hình tập dữ liệu và resume

**Quan trọng:** `RUN_NAME` phải **giữ nguyên** khi chạy lại cùng một batch. `GAMES` là **tổng số game_index mong muốn**, không phải số ván cần tạo thêm. Ví dụ: chạy 10 ván đầu với GAMES=10, sau đó đổi GAMES=50 cùng RUN_NAME sẽ tạo tiếp 40 ván.

Nếu thay đổi `MAX_PLIES`, seed, độ sâu hay agent, **đổi RUN_NAME**. Resume sẽ từ chối cấu hình khác để tránh trộn nhãn/dữ liệu.

In [ ]:
RED_AGENT = 'alphabeta'
BLACK_AGENT = 'alphabeta'
RED_DEPTH = 2
BLACK_DEPTH = 2
GAMES = 50
MAX_PLIES = 240
OPENING_PLIES = 2
NO_PROGRESS_LIMIT = 0
SEED = 42

RUN_NAME = 'alphabeta_depth2_seed42_max240_v1'  # giữ cố định khi resume!
WORKERS = 1  # thử 2 nếu Colab có >= 2 CPU và RAM đủ
PLY_PROGRESS_EVERY = 20  # log mỗi 20 ply khi WORKERS=1

OUTPUT_FILE = SAVE_DIR / f'{RUN_NAME}.jsonl'
CHECKPOINT_FILE = SAVE_DIR / f'{RUN_NAME}.jsonl.checkpoint.json'
print('Dataset:', OUTPUT_FILE)
print('Checkpoint:', CHECKPOINT_FILE)
print('Already exists:', OUTPUT_FILE.exists())

## 5. Chạy batch, theo dõi và tiếp tục

Mỗi ván bắt đầu sẽ in `[game X] started`; mỗi `PLY_PROGRESS_EVERY` lượt sẽ báo `ply/giới hạn`. Sau **mỗi ván hoàn tất**, in `[batch N/GAMES]` kèm trạng thái, thời gian ván, ETA rồi ghi JSONL và checkpoint. Có thể ngắt runtime và chạy lại; ván chưa hoàn thành sẽ chạy từ đầu. `WORKERS > 1` chạy nhiều ván độc lập song song; log ply được tắt để tránh lẫn thứ tự.

**Lưu ý:** Drive mount có thể đồng bộ trễ; không chạy đồng thời hai notebook ghi cùng RUN_NAME.

In [ ]:
command = [
    sys.executable, '-u', '-m', 'src', 'batch',
    '--red', RED_AGENT, '--black', BLACK_AGENT,
    '--red-depth', str(RED_DEPTH), '--black-depth', str(BLACK_DEPTH),
    '--games', str(GAMES), '--max-plies', str(MAX_PLIES),
    '--opening-plies', str(OPENING_PLIES),
    '--no-progress-limit', str(NO_PROGRESS_LIMIT),
    '--seed', str(SEED), '--output', str(OUTPUT_FILE),
    '--progress-every', '1',
    '--ply-progress-every', str(PLY_PROGRESS_EVERY),
    '--workers', str(WORKERS),
    '--resume',
]
print('Starting / resuming CPU batch generation...')
subprocess.run(command, cwd=str(BACKEND_DIR), check=True)

## 6. Xem tỷ lệ hòa, ván cắt ngắn và tốc độ thực tế

Nếu nhiều ván `repetition` hoặc `truncated`, chuyển lên máy mạnh hơn **không làm tăng chất lượng nhãn thắng/thua**. Hãy đa dạng hóa đối thủ và khai cuộc trước khi tăng số lượng.

In [ ]:
import collections
import json
import statistics

records = [json.loads(line) for line in OUTPUT_FILE.read_text(encoding='utf-8').splitlines() if line.strip()]
results = collections.Counter(
    (r['outcome']['status'], r['outcome']['reason']) for r in records
)
print('Records saved:', len(records))
print('Result breakdown:', dict(results))
if records:
    print('Average ply:', round(statistics.mean(r['ply_count'] for r in records), 2))
    print('Average seconds/game:', round(statistics.mean(r['elapsed_seconds'] for r in records), 2))
    print('Unique fingerprints:', len({r['fingerprint'] for r in records}))
    print('Full results:', [(r['game_index'], r['outcome']) for r in records])